# Oxigraph `dirLangString` equality crash

**Finding**: pyoxigraph's SPARQL `=` operator panics the whole process (a Rust
`unreachable!()`, not a catchable error) on *any* comparison involving an
RDF 1.2 `rdf:dirLangString`-typed term - a direction-tagged language string
(e.g. `"hi"@en--ltr`). This is **not** specific to the `STRLANGDIR()`
function, despite that being how it was first found - every scenario below
was tested directly against bare `pyoxigraph`, with no StarLayer code
involved, to isolate the real trigger.

Confirmed against:
- **Embedded**: `pyoxigraph` `0.5.11` (this notebook)
- **Remote/HTTP**: the standalone `oxigraph` server, `0.5.9` (crashes the
  server process itself - confirmed separately, not reproduced here since
  it requires a live container)

See `oxigraph-upstream-issues.md` (same folder) Issue #2
for the full writeup.

In [1]:
import pyoxigraph as ox
print(ox.__version__ if hasattr(ox, '__version__') else 'unknown')

0.5.11


## Baseline: reading/constructing a `dirLangString` value works fine

No comparison involved at all - plain construction via `STRLANGDIR()`, and
plain reading of a stored value. Both work exactly as expected.

In [2]:
store = ox.Store()
store.update('INSERT DATA { <http://example.org/a> <http://example.org/title> "hi"@en--ltr }')
print(store.dump(format=ox.RdfFormat.TURTLE, from_graph=ox.DefaultGraph()).decode())

result = list(store.query('SELECT (STRLANGDIR("hi", "en", "ltr") AS ?v) WHERE {}'))
print(result)

<http://example.org/a> <http://example.org/title> "hi"@en--ltr .

[<QuerySolution v=<Literal value=hi language=en direction=ltr>>]


In [3]:
store2 = ox.Store()
store2.update('INSERT DATA { <http://example.org/a> <http://example.org/title> "hi"@en--ltr }')
result = list(store2.query('SELECT ?t WHERE { ?s <http://example.org/title> ?t }'))
print(result)

[<QuerySolution t=<Literal value=hi language=en direction=ltr>>]


## `sameTerm()` also works fine

`sameTerm()` is exact RDF-term equality - a different SPARQL operator from
`=`, implemented without the per-datatype value-equality dispatch that
turns out to be where the bug actually lives.

In [4]:
store3 = ox.Store()
store3.update('INSERT DATA { <http://example.org/a> <http://example.org/title> "hi"@en--ltr }')
result = list(store3.query('SELECT ?s WHERE { ?s <http://example.org/title> ?t . FILTER(sameTerm(?t, ?t)) }'))
print(result)

[<QuerySolution s=<NamedNode value=http://example.org/a>>]


## The crash: `=` on *any* `dirLangString` term

Every variant below panics identically - self-comparison of a single
stored value, two independently stored values, two `STRLANGDIR()`-
constructed values, and the original comparison (stored value vs. a
`STRLANGDIR()`-constructed one) that first surfaced this. None of them
involve StarLayer code.

**These cells cannot be executed in this notebook's own kernel** - each
one is a hard process abort (`SIGABRT`), which would kill the kernel
running this notebook, not raise a catchable exception. Each was instead
run once, independently, as its own subprocess, and the real captured
output (stdout/stderr/exit code) is embedded below exactly as produced -
not hand-written or guessed.

### Self-equality on one stored value

In [ ]:
import pyoxigraph as ox
store = ox.Store()
store.update('INSERT DATA { <http://example.org/a> <http://example.org/title> "hi"@en--ltr }')
list(store.query('SELECT ?s WHERE { ?s <http://example.org/title> ?t . FILTER(?t = ?t) }'))

### Two independently stored values, compared

In [ ]:
import pyoxigraph as ox
store = ox.Store()
store.update('''
    INSERT DATA {
      <http://example.org/a> <http://example.org/title> "hi"@en--ltr .
      <http://example.org/b> <http://example.org/title> "hi"@en--ltr .
    }
''')
list(store.query('''
    SELECT ?s1 ?s2 WHERE {
      ?s1 <http://example.org/title> ?t1 .
      ?s2 <http://example.org/title> ?t2 .
      FILTER(?t1 = ?t2)
    }
'''))

### Two STRLANGDIR()-constructed values, compared (no stored data at all)

In [ ]:
import pyoxigraph as ox
store = ox.Store()
list(store.query('SELECT ?eq WHERE { BIND(STRLANGDIR("x","en","ltr") = STRLANGDIR("x","en","ltr") AS ?eq) }'))

### Original repro: a stored value vs. a STRLANGDIR()-constructed one

In [ ]:
import pyoxigraph as ox
store = ox.Store()
store.update('''
    INSERT DATA { <http://example.org/dir> <http://example.org/title>
      "HTML page"@en--ltr }
''')
list(store.query('''
    SELECT ?s WHERE {
      ?s <http://example.org/title> ?t .
      FILTER(?t = STRLANGDIR("HTML page", "en", "ltr"))
    }
'''))

## Confirmed Oxigraph-specific: Apache Jena Fuseki handles every case correctly

Requires a live local Fuseki test server
(`docker run -d --name fuseki-test -p 3030:3030 atomgraph/fuseki:6.1.0`, dataset
`starlayergraph`) - skips gracefully below if one isn't reachable, rather than
failing. Every scenario above that crashes Oxigraph - including the minimal
self-equality case - returns a correct result against Fuseki/ARQ, through
`StarLayerGraph`'s own native `backend="rdf-1.2"` code path (not bypassing
it). This rules out "the spec is ambiguous, engines reasonably disagree" -
Jena/ARQ implements `dirLangString` value-equality correctly.

In [1]:
import uuid
import requests
from rdflib import Namespace, URIRef

FUSEKI_BASE = "http://localhost:3030/starlayergraph"

try:
    _reachable = requests.get("http://localhost:3030/$/ping", timeout=2).status_code == 200
except Exception:
    _reachable = False

if not _reachable:
    print("Fuseki not reachable at localhost:3030 - skipping this section.")
else:
    from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore
    from starlayer.graph.graph.starlayer_graph import StarLayerGraph
    from starlayer.graph.model.dirlangstring import DirLangString

    EX = Namespace("http://example.org/")

    def fresh_graph():
        fresh_id = URIRef(f"http://example.org/g-{uuid.uuid4().hex}")
        store = SPARQLUpdateStore(
            query_endpoint=f"{FUSEKI_BASE}/query",
            update_endpoint=f"{FUSEKI_BASE}/update",
        )
        return StarLayerGraph(store=store, identifier=fresh_id, backend="rdf-1.2")

    g = fresh_graph()
    g.add((EX.a, EX.title, DirLangString("hi", "en", "ltr")))

    print("self-equality FILTER(?t = ?t):")
    print(list(g.query(f'''
        SELECT ?s WHERE {{
          GRAPH <{g.identifier}> {{ ?s <{EX}title> ?t . FILTER(?t = ?t) }}
        }}
    ''')))

    g2 = fresh_graph()
    g2.add((EX.a, EX.title, DirLangString("hi", "en", "ltr")))

    print("?t = STRLANGDIR(...) - the original Oxigraph repro:")
    print(list(g2.query(f'''
        SELECT ?s WHERE {{
          GRAPH <{g2.identifier}> {{
            ?s <{EX}title> ?t .
            FILTER(?t = STRLANGDIR("hi", "en", "ltr"))
          }}
        }}
    ''')))


self-equality FILTER(?t = ?t):
[(rdflib.term.URIRef('http://example.org/a'),)]
?t = STRLANGDIR(...) - the original Oxigraph repro:
[(rdflib.term.URIRef('http://example.org/a'),)]


## Conclusion

`sameTerm()` is a safe, usable comparison for `dirLangString` values today;
`=` (and therefore anything that relies on SPARQL value-equality for a
`dirLangString` - `DISTINCT`, `FILTER(?a = ?b)` style constraints, `VALUES`
filtering, etc. - not all individually tested here) is not, against either
the embedded `pyoxigraph` binding or a standalone `oxigraph` server, across
the two released versions checked (`0.5.9`, `0.5.11`). Apache Jena Fuseki
(ARQ) handles every one of these cases correctly, confirming this is an
Oxigraph-specific engine gap, not a genuinely ambiguous corner of the SPARQL
1.2 spec. Logged as Issue #2 in `oxigraph-upstream-issues.md`, pending an
upstream fix.